In [30]:
#%pip install itables

## Gold EV Charging Infrastructure by Postcode

The Gold transformation combines the Silver vehicle-registration and EV charging-location datasets into a postcode-level analytical table. It provides a consolidated view of registered electric vehicles, existing charging infrastructure and infrastructure coverage across Victoria for the selected reporting period.

The Gold table has one row per postcode and analysis period. A full outer join retains postcodes with registered EVs but no charging infrastructure, as well as charging locations without corresponding EV registrations.

The transformation includes:

- Aggregating total vehicles, EVs, non-EVs and unknown-fuel vehicles by postcode.
- Calculating EVs as a percentage of the registered vehicle fleet.
- Counting the number of distinct EV manufacturers represented.
- Aggregating public, DC fast, DC ultra-fast and AC destination charging sites.
- Summing charging plugs, bays and estimated installed charging capacity.
- Counting 24/7 public sites and charging-network operators.
- Calculating EVs per charging site, public DC site and public DC plug.
- Calculating public DC plugs per 100 registered EVs.
- Estimating the number of DC plugs required using the current benchmark of one plug per 100 EVs.
- Calculating postcode-level public DC plug shortfalls and surpluses.
- Classifying each postcode according to its charging coverage.
- Recording the registration and charging snapshot dates.
- Creating a deterministic postcode-period key.
- Adding a Gold processing timestamp.
- Reconciling EV and public DC plug totals against the two Silver source tables.
- Writing the result to the managed `gold_ev_charging_infrastructure_by_postcode` Delta table.

The initial MVP assumes that fuel code `E` identifies electric vehicles and uses manually configured snapshot dates. Future versions should use authoritative fuel-type and postcode reference tables to add canonical localities, local government areas, planning regions and statistical geography.

In [31]:
from itables import show

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# Retrieve the active Spark session or create one.
spark = SparkSession.builder.getOrCreate()


# -------------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------------

table_silver_vehicle_registrations = (
    "transport_planning.default."
    "silver_whole_fleet_vehicle_registrations_by_postcode"
)

table_silver_ev_charging_locations = (
    "transport_planning.default."
    "silver_ev_charging_locations"
)

table_gold_ev_charging_by_postcode = (
    "transport_planning.default."
    "gold_ev_charging_infrastructure_by_postcode"
)

analysis_period = "2026-Q2"
registration_snapshot_date = "2026-06-30"

# Update this value if a more precise charging snapshot date is available.
charging_snapshot_date = "2026-04-30"

# Provisional classification pending an authoritative fuel-code dimension.
electric_fuel_type_codes = ["E"]

# Current statewide benchmark: one public DC plug per 100 EVs.
benchmark_evs_per_public_dc_plug = 100


# -------------------------------------------------------------------------
# Read Silver tables
# -------------------------------------------------------------------------

vehicle_registration_df = spark.table(
    table_silver_vehicle_registrations
)

charging_location_df = spark.table(
    table_silver_ev_charging_locations
)

In [32]:
# -------------------------------------------------------------------------
# Aggregate vehicle registrations by postcode
# -------------------------------------------------------------------------

# Limit the MVP analysis to Victorian-format postcodes.
victorian_vehicle_df = vehicle_registration_df.filter(
    F.col("is_victorian_postcode") == True
)

is_electric_vehicle = F.col("fuel_type_code").isin(
    electric_fuel_type_codes
)

is_unknown_fuel = (
    F.col("fuel_type_code").isNull()
    | (F.col("fuel_type_code") == "UNKNOWN")
)

vehicle_postcode_df = (
    victorian_vehicle_df
    .groupBy("postcode")
    .agg(
        F.sum("registered_vehicle_count")
        .cast("long")
        .alias("total_registered_vehicle_count"),

        F.sum(
            F.when(
                is_electric_vehicle,
                F.col("registered_vehicle_count"),
            ).otherwise(F.lit(0))
        )
        .cast("long")
        .alias("registered_ev_count"),

        F.sum(
            F.when(
                ~is_electric_vehicle & ~is_unknown_fuel,
                F.col("registered_vehicle_count"),
            ).otherwise(F.lit(0))
        )
        .cast("long")
        .alias("registered_non_ev_count"),

        F.sum(
            F.when(
                is_unknown_fuel,
                F.col("registered_vehicle_count"),
            ).otherwise(F.lit(0))
        )
        .cast("long")
        .alias("unknown_fuel_vehicle_count"),

        F.countDistinct(
            F.when(
                is_electric_vehicle,
                F.col("vehicle_make_code"),
            )
        )
        .cast("long")
        .alias("registered_ev_make_count"),
    )
)

In [33]:
# -------------------------------------------------------------------------
# Aggregate charging infrastructure by postcode
# -------------------------------------------------------------------------

is_public = (
    F.lower(F.col("access_type"))
    .startswith("public")
)

is_dc_fast = (
    F.col("charger_category") == "DC fast"
)

is_dc_ultra_fast = (
    F.col("charger_category") == "DC ultra-fast"
)

is_public_dc = (
    is_public
    & (is_dc_fast | is_dc_ultra_fast)
)

is_ac_destination = (
    F.col("charger_category") == "AC destination"
)

is_available_24_7 = (
    F.lower(F.col("availability")) == "24/7"
)


charging_postcode_df = (
    charging_location_df
    .groupBy("postcode")
    .agg(
        # Geographic values are arrays because a postcode may contain
        # multiple localities or charging regions.
        F.sort_array(
            F.collect_set("suburb_locality")
        ).alias("charging_localities"),

        F.sort_array(
            F.collect_set("region")
        ).alias("charging_regions"),

        # Site measures
        F.countDistinct("location_id")
        .cast("long")
        .alias("charging_site_count"),

        F.countDistinct(
            F.when(
                is_public,
                F.col("location_id"),
            )
        )
        .cast("long")
        .alias("public_charging_site_count"),

        F.countDistinct(
            F.when(
                is_public_dc,
                F.col("location_id"),
            )
        )
        .cast("long")
        .alias("public_dc_site_count"),

        F.countDistinct(
            F.when(
                is_dc_fast,
                F.col("location_id"),
            )
        )
        .cast("long")
        .alias("dc_fast_site_count"),

        F.countDistinct(
            F.when(
                is_dc_ultra_fast,
                F.col("location_id"),
            )
        )
        .cast("long")
        .alias("dc_ultra_fast_site_count"),

        F.countDistinct(
            F.when(
                is_ac_destination,
                F.col("location_id"),
            )
        )
        .cast("long")
        .alias("ac_destination_site_count"),

        F.countDistinct(
            F.when(
                is_public & is_available_24_7,
                F.col("location_id"),
            )
        )
        .cast("long")
        .alias("public_24_7_site_count"),

        # Plug measures
        F.sum("charger_count")
        .cast("long")
        .alias("total_plug_count"),

        F.sum(
            F.when(
                is_public,
                F.col("charger_count"),
            ).otherwise(F.lit(0))
        )
        .cast("long")
        .alias("public_plug_count"),

        F.sum(
            F.when(
                is_public_dc,
                F.col("charger_count"),
            ).otherwise(F.lit(0))
        )
        .cast("long")
        .alias("public_dc_plug_count"),

        F.sum(
            F.when(
                is_dc_fast,
                F.col("charger_count"),
            ).otherwise(F.lit(0))
        )
        .cast("long")
        .alias("dc_fast_plug_count"),

        F.sum(
            F.when(
                is_dc_ultra_fast,
                F.col("charger_count"),
            ).otherwise(F.lit(0))
        )
        .cast("long")
        .alias("dc_ultra_fast_plug_count"),

        F.sum(
            F.when(
                is_ac_destination,
                F.col("charger_count"),
            ).otherwise(F.lit(0))
        )
        .cast("long")
        .alias("ac_destination_plug_count"),

        # Supporting measures
        F.sum("charging_bays")
        .cast("long")
        .alias("charging_bay_count"),

        F.countDistinct("network_operator")
        .cast("long")
        .alias("charging_operator_count"),

        # This is an estimate because max_power_kw may not represent
        # simultaneously deliverable site capacity.
        F.sum(
            F.col("charger_count")
            * F.col("max_power_kw")
        )
        .cast("double")
        .alias("estimated_installed_capacity_kw"),
    )
)

In [34]:
# -------------------------------------------------------------------------
# Combine vehicle and charging aggregates
# -------------------------------------------------------------------------

gold_df = (
    vehicle_postcode_df
    .join(
        charging_postcode_df,
        on="postcode",
        how="full",
    )
)


# Replace missing aggregate measures with zero.
count_columns = [
    "total_registered_vehicle_count",
    "registered_ev_count",
    "registered_non_ev_count",
    "unknown_fuel_vehicle_count",
    "registered_ev_make_count",
    "charging_site_count",
    "public_charging_site_count",
    "public_dc_site_count",
    "dc_fast_site_count",
    "dc_ultra_fast_site_count",
    "ac_destination_site_count",
    "public_24_7_site_count",
    "total_plug_count",
    "public_plug_count",
    "public_dc_plug_count",
    "dc_fast_plug_count",
    "dc_ultra_fast_plug_count",
    "ac_destination_plug_count",
    "charging_bay_count",
    "charging_operator_count",
]

gold_df = gold_df.fillna(
    0,
    subset=count_columns,
)

gold_df = gold_df.fillna(
    0.0,
    subset=["estimated_installed_capacity_kw"],
)

In [35]:
# -------------------------------------------------------------------------
# Calculate postcode-level coverage measures
# -------------------------------------------------------------------------

gold_df = (
    gold_df
    .withColumn(
        "analysis_period",
        F.lit(analysis_period),
    )
    .withColumn(
        "registration_snapshot_date",
        F.to_date(F.lit(registration_snapshot_date)),
    )
    .withColumn(
        "charging_snapshot_date",
        F.to_date(F.lit(charging_snapshot_date)),
    )
    .withColumn(
        "snapshot_date_difference_days",
        F.datediff(
            F.col("registration_snapshot_date"),
            F.col("charging_snapshot_date"),
        ),
    )

    # EV share of the registered fleet.
    .withColumn(
        "ev_share_pct",
        F.when(
            F.col("total_registered_vehicle_count") > 0,
            (
                F.col("registered_ev_count")
                / F.col("total_registered_vehicle_count")
            ) * F.lit(100.0),
        ),
    )

    # Charging infrastructure ratios.
    .withColumn(
        "evs_per_charging_site",
        F.when(
            F.col("charging_site_count") > 0,
            F.col("registered_ev_count")
            / F.col("charging_site_count"),
        ),
    )
    .withColumn(
        "evs_per_public_dc_site",
        F.when(
            F.col("public_dc_site_count") > 0,
            F.col("registered_ev_count")
            / F.col("public_dc_site_count"),
        ),
    )
    .withColumn(
        "evs_per_public_dc_plug",
        F.when(
            F.col("public_dc_plug_count") > 0,
            F.col("registered_ev_count")
            / F.col("public_dc_plug_count"),
        ),
    )
    .withColumn(
        "public_dc_plugs_per_100_evs",
        F.when(
            F.col("registered_ev_count") > 0,
            (
                F.col("public_dc_plug_count")
                / F.col("registered_ev_count")
            ) * F.lit(100.0),
        ),
    )

    # Required plugs at the current one-per-100-EV benchmark.
    .withColumn(
        "benchmark_required_dc_plugs",
        F.ceil(
            F.col("registered_ev_count")
            / F.lit(benchmark_evs_per_public_dc_plug)
        ).cast("long"),
    )

    # Shortfall is always zero or positive.
    .withColumn(
        "public_dc_plug_shortfall",
        F.greatest(
            F.col("benchmark_required_dc_plugs")
            - F.col("public_dc_plug_count"),
            F.lit(0),
        ).cast("long"),
    )

    # Surplus is always zero or positive.
    .withColumn(
        "public_dc_plug_surplus",
        F.greatest(
            F.col("public_dc_plug_count")
            - F.col("benchmark_required_dc_plugs"),
            F.lit(0),
        ).cast("long"),
    )

    .withColumn(
        "has_public_dc_charging",
        F.col("public_dc_plug_count") > 0,
    )

    # Assign an analysis-friendly coverage category.
    .withColumn(
        "charging_coverage_status",
        F.when(
            (F.col("registered_ev_count") == 0)
            & (F.col("public_dc_plug_count") == 0),
            F.lit("NO_EVS_OR_DC_CHARGING"),
        )
        .when(
            (F.col("registered_ev_count") == 0)
            & (F.col("public_dc_plug_count") > 0),
            F.lit("CHARGING_WITHOUT_REGISTERED_EVS"),
        )
        .when(
            (F.col("registered_ev_count") > 0)
            & (F.col("public_dc_plug_count") == 0),
            F.lit("NO_PUBLIC_DC_CHARGING"),
        )
        .when(
            F.col("public_dc_plug_count")
            < F.col("benchmark_required_dc_plugs"),
            F.lit("BELOW_CURRENT_BENCHMARK"),
        )
        .when(
            F.col("public_dc_plug_count")
            == F.col("benchmark_required_dc_plugs"),
            F.lit("MEETS_CURRENT_BENCHMARK"),
        )
        .otherwise(
            F.lit("ABOVE_CURRENT_BENCHMARK")
        ),
    )

    .withColumn(
        "gold_processed_at",
        F.current_timestamp(),
    )
)


# Create a deterministic Gold key.
gold_df = gold_df.withColumn(
    "gold_postcode_key",
    F.sha2(
        F.concat_ws(
            "||",
            F.col("analysis_period"),
            F.col("postcode"),
        ),
        256,
    ),
)

In [36]:
# -------------------------------------------------------------------------
# Select and order the Gold fields
# -------------------------------------------------------------------------

gold_df = gold_df.select(
    "gold_postcode_key",
    "analysis_period",
    "registration_snapshot_date",
    "charging_snapshot_date",
    "snapshot_date_difference_days",
    "postcode",
    "charging_localities",
    "charging_regions",

    # Vehicle measures
    "total_registered_vehicle_count",
    "registered_ev_count",
    "registered_non_ev_count",
    "unknown_fuel_vehicle_count",
    "registered_ev_make_count",
    "ev_share_pct",

    # Charging site measures
    "charging_site_count",
    "public_charging_site_count",
    "public_dc_site_count",
    "dc_fast_site_count",
    "dc_ultra_fast_site_count",
    "ac_destination_site_count",
    "public_24_7_site_count",

    # Charging plug and capacity measures
    "total_plug_count",
    "public_plug_count",
    "public_dc_plug_count",
    "dc_fast_plug_count",
    "dc_ultra_fast_plug_count",
    "ac_destination_plug_count",
    "charging_bay_count",
    "charging_operator_count",
    "estimated_installed_capacity_kw",

    # Coverage measures
    "evs_per_charging_site",
    "evs_per_public_dc_site",
    "evs_per_public_dc_plug",
    "public_dc_plugs_per_100_evs",
    "benchmark_required_dc_plugs",
    "public_dc_plug_shortfall",
    "public_dc_plug_surplus",
    "has_public_dc_charging",
    "charging_coverage_status",

    # Processing metadata
    "gold_processed_at",
)

In [37]:
# -------------------------------------------------------------------------
# Write the managed Gold Delta table
# -------------------------------------------------------------------------

(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(table_gold_ev_charging_by_postcode)
)

In [38]:
# -------------------------------------------------------------------------
# Verification and reconciliation
# -------------------------------------------------------------------------

gold_result_df = spark.table(
    table_gold_ev_charging_by_postcode
)

gold_row_count = gold_result_df.count()

duplicate_gold_keys = (
    gold_result_df
    .groupBy("gold_postcode_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

source_ev_total = (
    vehicle_postcode_df
    .agg(
        F.sum("registered_ev_count").alias("value")
    )
    .first()["value"]
)

gold_ev_total = (
    gold_result_df
    .agg(
        F.sum("registered_ev_count").alias("value")
    )
    .first()["value"]
)

source_public_dc_plug_total = (
    charging_postcode_df
    .agg(
        F.sum("public_dc_plug_count").alias("value")
    )
    .first()["value"]
)

gold_public_dc_plug_total = (
    gold_result_df
    .agg(
        F.sum("public_dc_plug_count").alias("value")
    )
    .first()["value"]
)


print(f"Gold postcode rows:        {gold_row_count:,}")
print(f"Duplicate Gold keys:       {duplicate_gold_keys:,}")
print(f"Source registered EVs:     {source_ev_total:,}")
print(f"Gold registered EVs:       {gold_ev_total:,}")
print(
    "EV totals match:           "
    f"{source_ev_total == gold_ev_total}"
)
print(
    f"Source public DC plugs:    "
    f"{source_public_dc_plug_total:,}"
)
print(
    f"Gold public DC plugs:      "
    f"{gold_public_dc_plug_total:,}"
)
print(
    "DC plug totals match:      "
    f"{source_public_dc_plug_total == gold_public_dc_plug_total}"
)


# Display the Gold schema.
gold_result_df.printSchema()


# Render a small interactive preview.
show(
    gold_result_df
    .orderBy(
        F.col("public_dc_plug_shortfall").desc(),
        F.col("registered_ev_count").desc(),
    )
    .limit(20)
    .toPandas()
)

Gold postcode rows:        745
Duplicate Gold keys:       0
Source registered EVs:     136,381
Gold registered EVs:       136,381
EV totals match:           True
Source public DC plugs:    1,000
Gold public DC plugs:      1,000
DC plug totals match:      True
root
 |-- gold_postcode_key: string (nullable = true)
 |-- analysis_period: string (nullable = true)
 |-- registration_snapshot_date: date (nullable = true)
 |-- charging_snapshot_date: date (nullable = true)
 |-- snapshot_date_difference_days: integer (nullable = true)
 |-- postcode: string (nullable = true)
 |-- charging_localities: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- charging_regions: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- total_registered_vehicle_count: long (nullable = true)
 |-- registered_ev_count: long (nullable = true)
 |-- registered_non_ev_count: long (nullable = true)
 |-- unknown_fuel_vehicle_count: long (nullable = true)
 |-- registered_ev

<!--| quarto-html-table-processing: none -->
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 

 
 
 
 

 
 
 
 

 
 
 
 

 
 
 
 
 
 
 
 
 Loading ITables v2.9.1 from the internet...
 (need help ?)
 
 
 
 
 
 🔒 ⓘ gold_postcode_key 
 analysis_period 
 registration_snapshot_date 
 charging_snapshot_date 
 snapshot_date_difference_days 
 postcode 
 charging_localities 
 charging_regions 
 total_registered_vehicle_count 
 registered_ev_count 
 registered_non_ev_count 
 unknown_fuel_vehicle_count 
 registered_ev_make_count 
 ev_share_pct 
 charging_site_count 
 public_charging_site_count 
 public_dc_site_count 
 dc_fast_site_count 
 dc_ultra_fast_site_count 
 ac_destination_site_count 
 public_24_7_site_count 
 total_plug_count 
 public_plug_count 
 public_dc_plug_count 
 dc_fast_plug_count 
 dc_ultra_fast_plug_count 
 ac_destination_plug_count 
 charging_bay_count 
 charging_operator_count 
 estimated_installed_capacity_kw 
 evs_per_charging_site 
 evs_per_public_dc_site 
 evs_per_public_dc_plug 
 public_dc_plugs_per_100_evs 
 benchmark_required_dc_plugs 
 public_dc_plug_shortfall 
 public_dc_plug_surplus 
 has_public_dc_charging 
 charging_coverage_status 
 gold_processed_at 
 
 51d602d220d6c82b0eec4023912de5c38f832eb984719caeb34e4d05d41cf1ad 2026-Q2 2026-06-30 2026-04-30 61 3150 [Glen Waverley] [Greater Melbourne] 46899 2440 44454 5 45 5.202670 10 10 3 3 0 7 5 28 28 6 6 0 22 28 5 884.0 244.000000 813.333333 406.666667 0.245902 25 19 0 True BELOW_CURRENT_BENCHMARK 2026-08-10 03:14:50.833930 
 35642d4fab710818d2257287c86f07a398fd77353a8aeff6a8e8e314c40b144b 2026-Q2 2026-06-30 2026-04-30 61 3029 [Hoppers Crossing] [Greater Melbourne] 124321 3119 121196 6 66 2.508828 8 7 4 3 1 4 5 24 22 14 8 6 10 24 4 2920.0 389.875000 779.750000 222.785714 0.448862 32 18 0 True BELOW_CURRENT_BENCHMARK 2026-08-10 03:14:50.833930 
 953f53eeaca583b6d833f8faa38f8e3f7197fee900fb654172e90a48630f37d8 2026-Q2 2026-06-30 2026-04-30 61 3064 [Craigieburn] [Greater Melbourne] 103448 1874 101567 7 48 1.811538 11 7 1 0 1 10 2 50 30 2 0 2 48 50 5 1356.0 170.363636 1874.000000 937.000000 0.106724 19 17 0 True BELOW_CURRENT_BENCHMARK 2026-08-10 03:14:50.833930 
 ccd8f188d85d00ba087084f006a22faebf31018a6109502f7dc83a8f77ae99bb 2026-Q2 2026-06-30 2026-04-30 61 3978 None None 55632 1699 53930 3 46 3.053998 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0.0 NaN NaN NaN 0.000000 17 17 0 False NO_PUBLIC_DC_CHARGING 2026-08-10 03:14:50.833930 
 1383b9501a22dc7ecd709429518bda6b061cc18d4606829170b920089462e44a 2026-Q2 2026-06-30 2026-04-30 61 3030 [Point Cook, Werribee] [Greater Melbourne] 94632 3142 91477 13 52 3.320230 25 23 5 3 2 20 17 100 90 16 10 6 84 100 5 3668.0 125.680000 628.400000 196.375000 0.509230 32 16 0 True BELOW_CURRENT_BENCHMARK 2026-08-10 03:14:50.833930 
 c83b9d1667a0ddcad3be689634639a85aa74ea3405f23bf1be096227a25b66f9 2026-Q2 2026-06-30 2026-04-30 61 3977 [Cranbourne] [Greater Melbourne] 97626 1807 95812 7 53 1.850941 10 7 2 1 1 8 6 30 20 4 2 2 26 30 4 1222.0 180.700000 903.500000 451.750000 0.221361 19 15 0 True BELOW_CURRENT_BENCHMARK 2026-08-10 03:14:50.833930 
 7c635d42a906b6541bbee492c024c6bf95b5fd1f28d28fa22bea3c7f1966a9f9 2026-Q2 2026-06-30 2026-04-30 61 3026 None None 17886 1416 16469 1 42 7.916806 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0.0 NaN NaN NaN 0.000000 15 15 0 False NO_PUBLIC_DC_CHARGING 2026-08-10 03:14:50.833930 
 31f5c13e418de5e57af77d0797024f42429768e106f3c8a44755b4e8f3764c43 2026-Q2 2026-06-30 2026-04-30 61 3170 None None 26292 1371 24920 1 42 5.214514 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0.0 NaN NaN NaN 0.000000 14 14 0 False NO_PUBLIC_DC_CHARGING 2026-08-10 03:14:50.833930 
 6104e0f2c843e891718294429e9a6c8316eaf75d1bd1ba02162edebcf7262a15 2026-Q2 2026-06-30 2026-04-30 61 3023 None None 51125 1091 50030 4 45 2.133985 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0.0 NaN NaN NaN 0.000000 11 11 0 False NO_PUBLIC_DC_CHARGING 2026-08-10 03:14:50.833930 
 a6205519100ba17508f841707d2a843da86af7180b5461263fae43d14ef907c0 2026-Q2 2026-06-30 2026-04-30 61 3152 None